# Tactical Fingerprints — Google Colab

Runs entirely from Google Drive. No file transfer needed.

**Expected Drive layout:**
```
MyDrive/
  Football_Events_SDS/
    Data/
      events/   events_England.json  ... (7 files)
      matches/  matches_England.json ... (7 files)
      players.json, teams.json, coaches.json, referees.json
      playerank.json, eventid2name.csv, tags2name.csv
```

## 1. Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Configure paths ─────────────────────────────────────────────────────────
DRIVE_DATA = '/content/drive/MyDrive/Football_Events_SDS/Data'
REPO_DIR   = '/content/Football-Analytics'

# Point the pipeline at your Drive folder — no copying needed
os.environ['WYSCOUT_RAW_DIR'] = DRIVE_DATA

# Competitions to analyse (remove any you don't need for speed)
COMPETITIONS = [
    'England', 'Spain', 'Italy', 'France', 'Germany',
    'European_Championship', 'World_Cup'
]

N_CLUSTERS = 5

In [ ]:
# Clone the repo (only needed once per Colab session)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yasarkerem/Football-Analytics.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Load data

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

from src.data.loader import load_events, load_matches, load_teams, load_players

events_df  = load_events(COMPETITIONS)
matches_df = load_matches(COMPETITIONS)
teams_df   = load_teams()
players_df = load_players()

print(f'Events:  {len(events_df):,}')
print(f'Matches: {len(matches_df):,}')
events_df.head()

## 3. Preprocess

In [ ]:
from src.data.preprocessor import preprocess, filter_passes

events_clean = preprocess(events_df, matches_df)
print(f'Clean events: {len(events_clean):,}')
events_clean['eventName'].value_counts()

## 4. Event features

In [ ]:
from src.features.event_features import extract_event_features

event_feats = extract_event_features(events_clean)
print(f'Feature rows: {len(event_feats):,}')
event_feats.describe()

## 5. Passing networks

In [ ]:
from src.network.builder import build_all_networks
from src.network.metrics import compute_network_metrics

passes      = filter_passes(events_clean, accurate_only=True)
networks    = build_all_networks(passes)
net_metrics = compute_network_metrics(networks)

print(f'Networks built: {len(networks):,}')
net_metrics.describe()

In [ ]:
# Visualise one team's passing network
import networkx as nx

sample_key = list(networks.keys())[0]
G = networks[sample_key]

fig, ax = plt.subplots(figsize=(8, 6))
weights = [G[u][v]['weight'] for u, v in G.edges()]
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos=pos, ax=ax,
                 width=[w * 0.3 for w in weights],
                 node_size=300, font_size=7,
                 edge_color='steelblue', node_color='#FF6B35', alpha=0.85)
ax.set_title(f'Passing Network — matchId={sample_key[0]}, teamId={sample_key[1]}')
plt.tight_layout()

## 6. Team profiles

In [ ]:
from src.features.aggregator import aggregate_team_profiles, build_fingerprint_matrix

combined = event_feats.merge(net_metrics, on=['matchId', 'teamId'], how='inner')
profiles = aggregate_team_profiles(combined, matches_df)
profiles = profiles.merge(teams_df[['teamId', 'name']], on='teamId', how='left')

print(f'Team profiles: {len(profiles):,}')
profiles[['name', 'competition', 'win_rate', 'points_per_match',
           'pass_accuracy', 'density', 'centralization'
          ]].sort_values('win_rate', ascending=False).head(15)

## 7. PCA + optimal cluster count

In [ ]:
from src.clustering.fingerprints import scale_features, run_pca, find_optimal_clusters
from src.visualization.plots import plot_pca_variance, plot_silhouette_curve

X, y = build_fingerprint_matrix(profiles)
feature_cols = [c for c in X.columns if c not in {'teamId', 'competition'}]

X_raw    = X[feature_cols].fillna(0).values
X_scaled, scaler = scale_features(X_raw)
X_pca, pca       = run_pca(X_scaled, n_components=10)

plot_pca_variance(np.cumsum(pca.explained_variance_ratio_))
plt.show()

eval_df = find_optimal_clusters(X_pca, k_range=range(2, 9))
plot_silhouette_curve(eval_df)
plt.show()
print(eval_df.to_string(index=False))

## 8. Tactical Fingerprints

In [ ]:
from src.clustering.fingerprints import build_fingerprint_report
from src.visualization.plots import (
    plot_cluster_scatter, plot_fingerprint_radar, plot_outcome_by_fingerprint
)

full_profiles = profiles.merge(
    y[['teamId', 'competition', 'win_rate', 'points_per_match', 'goalDiff']],
    on=['teamId', 'competition'], how='left'
)

result = build_fingerprint_report(
    profiles=full_profiles,
    feature_cols=feature_cols,
    n_clusters=N_CLUSTERS,
    use_umap=True,
)

labelled = result['profiles_labelled']
labelled = labelled.merge(teams_df[['teamId', 'name']], on='teamId', how='left')

plot_cluster_scatter(labelled, teams_df=teams_df)
plt.show()

In [ ]:
radar_features = ['pass_accuracy', 'long_ball_ratio', 'cross_ratio',
                  'smart_pass_ratio', 'density', 'centralization',
                  'pass_att_third', 'counter_rate']
plot_fingerprint_radar(labelled, feature_cols=radar_features)
plt.show()

## 9. Outcomes vs Fingerprints

In [ ]:
plot_outcome_by_fingerprint(labelled, metric='win_rate')
plt.show()
plot_outcome_by_fingerprint(labelled, metric='points_per_match')
plt.show()

In [ ]:
# Summary table
summary = labelled.groupby('fingerprint').agg(
    n_teams=('teamId', 'nunique'),
    win_rate=('win_rate', 'mean'),
    points_per_match=('points_per_match', 'mean'),
    pass_accuracy=('pass_accuracy', 'mean'),
    long_ball_ratio=('long_ball_ratio', 'mean'),
    cross_ratio=('cross_ratio', 'mean'),
    density=('density', 'mean'),
    centralization=('centralization', 'mean'),
).round(3)
print(summary.to_string())

In [ ]:
# Teams per fingerprint
for fp_id, grp in labelled.groupby('fingerprint'):
    print(f'\n── Fingerprint {fp_id} ({len(grp)} teams) ──')
    print(', '.join(grp['name'].dropna().unique()[:20]))

## 10. Save results to Drive

In [ ]:
import pathlib

OUT_DIR = pathlib.Path('/content/drive/MyDrive/Football_Events_SDS/Results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

labelled.to_csv(OUT_DIR / 'fingerprints.csv', index=False)
labelled.to_parquet(OUT_DIR / 'fingerprints.parquet', index=False)

# Save plots
from src.visualization.plots import (
    plot_cluster_scatter, plot_fingerprint_radar,
    plot_outcome_by_fingerprint, plot_pca_variance, plot_silhouette_curve
)
plot_cluster_scatter(labelled, teams_df=teams_df,
                     save_path=str(OUT_DIR / 'fingerprint_scatter.png'))
plot_fingerprint_radar(labelled, feature_cols=radar_features,
                       save_path=str(OUT_DIR / 'fingerprint_radar.png'))
plot_outcome_by_fingerprint(labelled, metric='win_rate',
                             save_path=str(OUT_DIR / 'win_rate_by_fp.png'))
plot_pca_variance(np.cumsum(pca.explained_variance_ratio_),
                  save_path=str(OUT_DIR / 'pca_variance.png'))

print(f'Results saved to {OUT_DIR}')